In [1]:
!pip install wittgenstein pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wittgenstein: filename=wittgenstein-0.3.5-py3-none-any.whl size=98989 sha256=57d4f748b666469dc6ce9521ae7aa6f53f0a556930a4bec8f28f0d306f1125ab
  Stored in directory: /root/.cache/pip/wheels/03/8e/f7/4afc64184996e67ef74ebefa7d74b324534a079fc3462242aa
Successfully built wittgenstein


In [2]:
import numpy as np
import pandas as pd
import wittgenstein as lw
from sklearn.datasets import load_iris

# ---- Step 1: Load dataset (Iris for demo) ---
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

df = pd.concat([X, y.rename("target")], axis=1)
print("Dataset head:\n", df.head())

# ---- Step 2: RIPPER Algorithm (via wittgenstein) ---
model = lw.RIPPER()
model.fit(X, y, pos_class=0)

print("\n=== RIPPER Rules ===")
print(model.ruleset_)

# ---- Step 3: Simplified FOIL Implementation ---
df_bin = df.copy()
df_bin["target"] = (df_bin["target"] == 0).astype(int)  # 1 = setosa, 0 = others

attributes = list(X.columns)


def foil_gain(pos_before, neg_before, pos_after, neg_after):
    # FOIL information gain formula using log2 base
    if pos_after == 0:
        return -1e9
    p1 = pos_before / (pos_before + neg_before)
    p2 = pos_after / (pos_after + neg_after)
    if p2 <= p1:
        return -1e9  # No improvement in precision
    return pos_after * (np.log2(p2) - np.log2(p1))


def foil(df_input, target_col="target"):
    df_current = df_input.copy()
    rules = []
    pos_total = df_current[target_col].sum()
    neg_total = len(df_current) - pos_total

    # Outer loop: cover all positive examples
    while pos_total > 0:
        rule = []
        pos_rem, neg_rem = pos_total, neg_total
        covered = df_current.copy()

        # Inner loop: grow a rule until it covers no negative examples
        while neg_rem > 0:
            best_gain = -1e9
            best_attr = None
            best_val = None
            best_subset = None

            for attr in attributes:
                for val in df_current[attr].unique():
                    # Filter subset based on the candidate literal
                    subset = covered[covered[attr] == val]
                    pos_after = subset[target_col].sum()
                    neg_after = len(subset) - pos_after

                    gain = foil_gain(pos_rem, neg_rem, pos_after, neg_after)

                    if gain > best_gain:
                        best_gain, best_attr, best_val, best_subset = (
                            gain,
                            attr,
                            val,
                            subset,
                        )

            if best_attr is None or best_gain <= 0:
                break  # Stop growing if no improvement can be made

            rule.append((best_attr, best_val))
            covered = best_subset
            pos_rem = covered[target_col].sum()
            neg_rem = len(covered) - pos_rem

        # Save rule and remove covered positive instances from remaining pool
        if len(rule) > 0:
            rules.append(rule)
            # Remove instances that perfectly match the built rule
            df_current = df_current.drop(covered[covered[target_col] == 1].index)
        else:
            break  # Prevent infinite loop if no rule can be generated

        pos_total = df_current[target_col].sum()

    return rules


print("\n=== Custom FOIL Rules ===")
foil_rules = foil(df_bin)
for idx, r in enumerate(foil_rules):
    rule_str = " AND ".join([f"({attr} == {val})" for attr, val in r])
    print(f"Rule {idx+1}: IF {rule_str} THEN Target = 1")


Dataset head:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

=== RIPPER Rules ===
[[petalwidth(cm)=<0.2] V [petalwidth(cm)=0.2-0.4] V [petallength(cm)=1.5-1.7]]

=== Custom FOIL Rules ===
Rule 1: IF (petal width (cm) == 0.2) THEN Target = 1
Rule 2: IF (petal width (cm) == 0.4) THEN Target = 1
Rule 3: IF (petal width (cm) == 0.3) THEN Target = 1
Rule 4: IF (petal width (cm) == 0.1) THEN Target = 1
Rule 5: IF (sepal width (cm) == 3.5) THEN Target = 1
Rule 6: IF (petal length (cm) == 1.7) THEN Ta